# 07 — Crow Search Algorithm (CSA) Feature Selection

Binary Crow Search Algorithm (BCSA) is used here because feature selection is a yes/no decision for every feature. The separate continuous CSA demonstration is intentionally omitted because it is not part of the experiment.


In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "utils").exists():
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))

import random
import numpy as np


## Binary Crow Search Algorithm

In [2]:
def sigmoid(x):
    x = np.clip(x, -20, 20)
    return 1.0 / (1.0 + np.exp(-x))


def run_bcsa(
    obj_func,
    n_features,
    pop_size=10,
    iterations=20,
    awareness_probability=0.1,
    flight_length=2.0,
):
    # Each crow position is a binary feature mask.
    positions = np.random.randint(
        0,
        2,
        size=(pop_size, n_features),
    )

    # A valid mask must select at least one feature.
    for position in positions:
        if position.sum() == 0:
            position[np.random.randint(n_features)] = 1

    # Each crow remembers the best position it has personally visited.
    memories = positions.copy()
    memory_scores = np.array([
        obj_func(position)
        for position in memories
    ])

    convergence = []

    for _ in range(iterations):
        candidates = np.empty_like(positions)

        for i in range(pop_size):
            possible = np.delete(np.arange(pop_size), i)
            followed = np.random.choice(possible)

            # If the followed crow is unaware, move toward its memory.
            if np.random.rand() >= awareness_probability:
                r = np.random.rand()
                continuous_position = (
                    positions[i]
                    + r
                    * flight_length
                    * (memories[followed] - positions[i])
                )
            else:
                # If it notices, it deceives the follower into a random move.
                continuous_position = np.random.uniform(
                    -1,
                    1,
                    size=n_features,
                )

            probabilities = sigmoid(continuous_position)
            candidate = (
                np.random.rand(n_features) < probabilities
            ).astype(int)

            if candidate.sum() == 0:
                candidate[np.random.randint(n_features)] = 1

            candidates[i] = candidate

        positions = candidates
        current_scores = np.array([
            obj_func(position)
            for position in positions
        ])

        improved = current_scores < memory_scores
        memories[improved] = positions[improved]
        memory_scores[improved] = current_scores[improved]

        convergence.append(float(memory_scores.min()))

    best_index = np.argmin(memory_scores)

    return (
        memories[best_index].copy(),
        float(memory_scores[best_index]),
        convergence,
    )


## Run the complete feature-selection experiment

The shared experiment runner supplies the configured datasets, classifiers,
optimizer seeds, population size, and iteration count. Feature selection uses
the validation set. The test set is evaluated only after the final mask has
been selected.


In [3]:
from utils.experiments import run_feature_selector


def csa_runner(
    objective,
    n_features,
    pop_size,
    iterations,
):
    return run_bcsa(
        obj_func=objective,
        n_features=n_features,
        pop_size=pop_size,
        iterations=iterations,
        awareness_probability=0.1,
        flight_length=2.0,
    )


csa_results = run_feature_selector("CSA", csa_runner)
csa_results.tail()


Saved: ('breast', 'svm', 0)


Saved: ('breast', 'svm', 1)


Saved: ('breast', 'svm', 2)


Saved: ('breast', 'svm', 3)


Saved: ('breast', 'svm', 4)


...

Saved: ('heart', 'xgboost', 1)


Saved: ('heart', 'xgboost', 2)


Saved: ('heart', 'xgboost', 3)


Saved: ('heart', 'xgboost', 4)


,Dataset,Classifier,Algorithm,Seed,ValidationFitness,Accuracy,Precision,Recall,F1,ROC_AUC,Features,SelectionRuntime,TestRuntime,SelectedFeatureNames,MaskFile,ConvergenceFile
25,heart,xgboost,CSA,0,0.125570,0.766304,0.804124,0.764706,0.783920,0.882473,18,21.590256,0.094589,"[""chol"", ""thalch"", ""oldpeak"", ""sex_Female"", ""s...",results/final/artifacts/heart__xgboost__csa__s...,results/final/artifacts/heart__xgboost__csa__s...
26,heart,xgboost,CSA,1,0.124770,0.793478,0.840426,0.774510,0.806122,0.890842,16,21.226908,0.093761,"[""chol"", ""thalch"", ""oldpeak"", ""ca"", ""sex_Femal...",results/final/artifacts/heart__xgboost__csa__s...,results/final/artifacts/heart__xgboost__csa__s...
27,heart,xgboost,CSA,2,0.132150,0.782609,0.829787,0.764706,0.795918,0.896581,21,21.172790,0.094930,"[""age"", ""chol"", ""oldpeak"", ""ca"", ""sex_Female"",...",results/final/artifacts/heart__xgboost__csa__s...,results/final/artifacts/heart__xgboost__csa__s...
28,heart,xgboost,CSA,3,0.114409,0.815217,0.846939,0.813725,0.830000,0.884326,17,21.486720,0.092516,"[""chol"", ""thalch"", ""oldpeak"", ""ca"", ""sex_Femal...",results/final/artifacts/heart__xgboost__csa__s...,results/final/artifacts/heart__xgboost__csa__s...
29,heart,xgboost,CSA,4,0.125570,0.793478,0.840426,0.774510,0.806122,0.892037,18,21.334041,0.094596,"[""chol"", ""thalch"", ""oldpeak"", ""ca"", ""sex_Male""...",results/final/artifacts/heart__xgboost__csa__s...,results/final/artifacts/heart__xgboost__csa__s...
